In [ ]:
import re, pandas as pd
from pathlib import Path

base = Path.cwd().resolve().parent; data_dir, out_dir = base/"data", base/"output"
lines = (data_dir/"GSE164641_series_matrix.txt").read_text(encoding="utf-8", errors="ignore").splitlines()
gsm = re.findall(r"GSM\d+", next(l for l in lines if l.startswith("!Sample_geo_accession")))
cats = re.findall(r"risk category: ([A-Za-z]+)", next(l for l in lines if "risk category" in l.lower()))
meta = pd.DataFrame({"sample_id": gsm[:len(cats)], "condition": cats})

counts = pd.read_csv(data_dir/"GSE164641_raw_counts_GRCh38.p13_NCBI.tsv", sep="\t", index_col=0)
meta = meta.set_index("sample_id").loc[counts.columns].rename_axis("sample_id").reset_index()

annot = pd.read_csv(data_dir/"Human.GRCh38.p13.annot.tsv", sep="\t", dtype=str).assign(GeneID=lambda d: d["GeneID"].str.split(".").str[0])

counts.to_csv(out_dir/"counts_for_edgeR.tsv", sep="\t")
meta.to_csv(out_dir/"metadata_for_edgeR.tsv", sep="\t", index=False)

print(counts.shape, "\n", meta.head(), "\n", annot[["GeneID","Symbol"]].head())

In [ ]:
from rpy2.robjects import r

r(f'''
library(edgeR)
counts <- read.delim("{counts_path.as_posix()}", row.names=1, check.names=FALSE)
meta <- read.delim("{meta_path.as_posix()}", stringsAsFactors=FALSE)
common <- intersect(colnames(counts), meta$sample_id)
if (length(common) == 0) stop("No overlapping sample IDs.")
counts <- counts[, common, drop=FALSE]
meta <- meta[match(common, meta$sample_id), , drop=FALSE]
group <- factor(trimws(meta$condition))
if (nlevels(group) < 2) stop(paste("Need 2 groups, got:", paste(levels(group), collapse=", ")))
y <- DGEList(counts=counts, group=group)
keep <- filterByExpr(y, group=group)
y <- y[keep, , keep.lib.sizes=FALSE]; y <- calcNormFactors(y)
design <- model.matrix(~group)
y <- estimateDisp(y, design); fit <- glmFit(y, design); lrt <- glmLRT(fit, coef=2)
res <- topTags(lrt, n=Inf)$table; res$GeneID <- rownames(res)
write.csv(res, "{(out_dir / 'edgeR_results_rpy2.csv').as_posix()}", row.names=FALSE)
''')

print("edgeR finished")

In [ ]:
res = pd.read_csv(out_dir / "edgeR_results_rpy2.csv")
res["GeneID"] = res["GeneID"].astype(str).str.split(".").str[0]
res = res.merge(annot[["GeneID", "Symbol"]], on="GeneID", how="left")
sig = res[(res["FDR"] < 0.05) & (res["logFC"].abs() > 1)].copy()

res.to_csv(out_dir / "edgeR_results_annotated_rpy2.csv", index=False)
sig.to_csv(out_dir / "edgeR_significant_genes_annotated_rpy2.csv", index=False)

print("Total genes:", len(res))
print("Significant genes:", len(sig))
sig.head()